In [ ]:
!pip install dedupe

In [1]:
import pandas as pd
import dedupe as dp
import os
import csv
import logging
import optparse
import re
from unidecode import unidecode
import time

In [2]:
from dedupe.variables import String

In [3]:
def preProcess(column):
    column = unidecode(column)
    column = re.sub("  +", " ", column)
    column = re.sub("\n", " ", column)
    column = column.strip().strip('"').strip("'").lower().strip()
    if not column:
        column = None
    return column

In [4]:
def readData(filename):
    data_d = {}
    with open(filename, encoding='utf-8-sig') as f:
        reader = csv.DictReader(f)
        id = 0
        for row in reader:
            clean_row = {(k, preProcess(v)) for (k, v) in row.items()}
            row_id = id
            data_d[row_id] = dict(clean_row)
            id += 1
    return data_d

In [ ]:
input_file = r"E:\Dedupe_and_Link_Datasets\Archive_Train\MetaLatin.csv"
    output_file = "MetaLatin_op.csv"
    settings_file = "csv_example_learned_settings_Archive"
    training_file = "csv_example_training_Archive.json"

    input_file = r"E:\Datasets\Testing_set\Testingset3.csv"
    output_file = "Testingset3_op.csv"
    settings_file = "csv_example_learned_settings_Archive"
    training_file = "csv_example_training_Archive.json"

In [41]:
if __name__ == "__main__":
    # Start timing
    start_time = time.time()
    
    input_file = "MetaLatin.csv"
    output_file = "MetaLatin_op.csv"
    settings_file = "Archive_Latin_learned_settings"
    training_file = "Archive_Latin_training.json"

    
    print("importing data ...")
    data_d = readData(input_file)
    if os.path.exists(settings_file):
        print("reading from", settings_file)
        with open(settings_file, "rb") as f:
            deduper = dp.StaticDedupe(f)
    else:
        print("initializing fields")
        fields = [
            dp.variables.String("creator"),
            dp.variables.String("title"),
            dp.variables.String("identifier"),
            dp.variables.String("publicdate"),
            dp.variables.Exact("year")
        ]
        print("start deduping fields")
        deduper = dp.Dedupe(fields)
        print("start writing training and setting")
        if os.path.exists(training_file):
            print("reading labeled examples from ", training_file)
            with open(training_file, "rb") as f:
                deduper.prepare_training(data_d, f)
        else:
            deduper.prepare_training(data_d)

    
            print("starting active labeling...")
    
            dp.console_label(deduper)
        
            deduper.train()
        
            with open(training_file, "w") as tf:
                deduper.write_training(tf)
        
            with open(settings_file, "wb") as sf:
                deduper.write_settings(sf)

    
    print("clustering...")
    clustered_dupes = deduper.partition(data_d, 0.5)
    
    print("# duplicate sets", len(clustered_dupes))


    cluster_membership = {}
    for cluster_id, (records, scores) in enumerate(clustered_dupes):
        for record_id, score in zip(records, scores):
            cluster_membership[record_id] = {
                "Cluster ID": cluster_id,
                "confidence_score": score,
            }
    
    with open(output_file, "w", encoding='utf-8-sig', newline='') as f_output, open(input_file, encoding='utf-8-sig') as f_input:
    
        reader = csv.DictReader(f_input)
        fieldnames = ["Cluster ID", "confidence_score"] + reader.fieldnames

        writer = csv.DictWriter(f_output, fieldnames=fieldnames)
        writer.writeheader()

        for row in reader:
            row_id = int(row["index"])
            row.update(cluster_membership[row_id])
            writer.writerow(row)

    # Stop timing
    end_time = time.time()
    
    # Calculate the total time needed to run the script
    total_time = end_time - start_time
    
    print(f"Total time needed to run the script: {total_time:.2f} seconds")

importing data ...
reading from Archive_Latin_learned_settings
clustering...


A component contained 44628 elements. Components larger than 30000 are re-filtered. The threshold for this filtering is 0.0009679235434766553
A component contained 44609 elements. Components larger than 30000 are re-filtered. The threshold for this filtering is 0.0026267734828352967
A component contained 44075 elements. Components larger than 30000 are re-filtered. The threshold for this filtering is 0.007108241153770778
A component contained 42468 elements. Components larger than 30000 are re-filtered. The threshold for this filtering is 0.019089061486196254
A component contained 38544 elements. Components larger than 30000 are re-filtered. The threshold for this filtering is 0.05024156338425962


# duplicate sets 42014
Total time needed to run the script: 442.67 seconds


In [ ]:
# Load the two CSV files
# df_a = pd.read_csv(r'E:\Datasets\Hathi_Catalogue\ft_pd_data_filtered_0.csv',header=0)
# df_b = pd.read_csv(r'E:\Datasets\Gallica Metadata\TGB_metadata.csv',header=0)

df_test_1 = pd.read_csv(r'E:\Datasets\Testing_set\Testingset1.csv',header=0)
df_test_2 = pd.read_csv(r'E:\Datasets\Testing_set\Testingset2.csv',header=0)

In [ ]:
input_file = r"E:\Datasets\Testing_set\Testingset1.csv"
output_file = "Testingset1_op.csv"
settings_file = "csv_example_learned_settings"
training_file = "csv_example_training.json"

In [ ]:
print("importing data ...")
data_d = readData(input_file)

In [ ]:
if os.path.exists(settings_file):
    print("reading from", settings_file)
    with open(settings_file, "rb") as f:
        deduper = dedupe.StaticDedupe(f)
else:
    fields = [
        dp.variables.String("author"),
        dp.variables.String("title"),
        dp.variables.Exact("oclc_num"),
    ]
    deduper = dp.Dedupe(fields)
    if os.path.exists(training_file):
        print("reading labeled examples from ", training_file)
        with open(training_file, "rb") as f:
            deduper.prepare_training(data_d, f)
    else:
        deduper.prepare_training(data_d)

In [ ]:
print("starting active labeling...")

dp.console_label(deduper)

deduper.train()

with open(training_file, "w") as tf:
    deduper.write_training(tf)

with open(settings_file, "wb") as sf:
    deduper.write_settings(sf)

In [ ]:
print("clustering...")
clustered_dupes = deduper.partition(data_d, 0.5)

print("# duplicate sets", len(clustered_dupes))

In [ ]:
cluster_membership = {}
for cluster_id, (records, scores) in enumerate(clustered_dupes):
    for record_id, score in zip(records, scores):
        cluster_membership[record_id] = {
            "Cluster ID": cluster_id,
            "confidence_score": score,
        }

with open(output_file, "w") as f_output, open(input_file) as f_input:

    reader = csv.DictReader(f_input)
    fieldnames = ["Cluster ID", "confidence_score"] + reader.fieldnames

    writer = csv.DictWriter(f_output, fieldnames=fieldnames)
    writer.writeheader()
    id = 0
    for row in reader:
        row_id = id
        row.update(cluster_membership[row_id])
        writer.writerow(row)
        id += 1

In [ ]:
#define dedupe field
fields = [String("title"),String("author"),String("oclc_num")]

In [ ]:
deduper = dp.Dedupe(fields)

In [ ]:
if os.path.exists(training_file):
    print("reading labeled examples from ", training_file)
    with open(training_file, "rb") as f:
        deduper.prepare_training(data_d, f)
else:
    deduper.prepare_training(data_d)